# Financial Friction Observatory

## Task 1: Data Exploration

### Objective
Understand the structure, size, completeness and
quality of the CFPB consumer complaint dataset.

### Data Sources
- 2023.csv
- 2024.csv
- 2025.csv

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import  Path

In [3]:
files = Path(r'C:\Users\Admin\OneDrive\Desktop\project\p3\data\raw').glob('*.csv')

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481


## 2. Dataset Structure

In this section, we examine:
- Number of rows and columns
- Column names
- Data types
- Non-null values
- Memory consumption

In [4]:
def details():
    print('-'*40)
    print('Number of Rows:', len(df))
    print('')
    print('Number of columns:', df.shape[1])
    print('')
    print('Name of Columns:','\n',df.columns)
    print('')
    print('Data types:','\n',df.dtypes)
    print('')
    print('DataFrame Information:')
    print( df.info(memory_usage="deep"))
    print('')
    print('Shape of DataFrame:', df.shape)
    print('')
    print('Memory Usage:')
    print(df.memory_usage(deep=True).sum() / (1024 ** 2), "MB")
    print('-'*40)

details()

----------------------------------------
Number of Rows: 186889

Number of columns: 15

Name of Columns: 
 Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Company public response', 'Company', 'State', 'ZIP code', 'Tags',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Complaint ID'],
      dtype='str')

Data types: 
 Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Company public response           str
Company                           str
State                             str
ZIP code                          str
Tags                              str
Submitted via                     str
Date sent to company              str
Company response to consumer      str
Timely response?                  str
Complaint ID                    int64
dtype: object

Data

## 3. Missing Value Analysis

Investigate missing values in each column to understand
the completeness of the dataset.

In [5]:
def missing():
    print('-'*40)
    print(df.isna().sum())
    print()

    columns = [
    'Tags',
    'Company public response',
    'Sub-issue',
    'State'
]

    for col in columns:
        print('-'*50)
        print(f'columns: {col}')
        print('Count',df[col].isna().sum())
        print('percentage:',df[col].isna().sum()/len(df) * 100,'%')

    print('-'*40)
    
    

missing()

----------------------------------------
Date received                        0
Product                              0
Sub-product                          5
Issue                                5
Sub-issue                          122
Company public response          92984
Company                              0
State                             1654
ZIP code                             0
Tags                            155015
Submitted via                        0
Date sent to company                 0
Company response to consumer         0
Timely response?                     0
Complaint ID                         0
dtype: int64

--------------------------------------------------
columns: Tags
Count 155015
percentage: 82.94495663201151 %
--------------------------------------------------
columns: Company public response
Count 92984
percentage: 49.75359705493636 %
--------------------------------------------------
columns: Sub-issue
Count 122
percentage: 0.06527939044031485 %
--------

In [6]:
columns = [
    'Tags',
    'Company public response',
    'Company response to consumer',
    'Timely response?'
]

for col in columns:
    print(f"\n{'-'*50}")
    # print(f"Column: {col}")
    print('column:', df[col].value_counts(dropna=False))


--------------------------------------------------
column: Tags
NaN                              155015
Servicemember                     15781
Older American                    12726
Older American, Servicemember      3367
Name: count, dtype: int64

--------------------------------------------------
column: Company public response
NaN                                                                                                                        92984
Company has responded to the consumer and the CFPB and chooses not to provide a public response                            76886
Company believes it acted appropriately as authorized by contract or law                                                   16232
Company believes the complaint provided an opportunity to answer consumer's questions                                        319
Company believes complaint represents an opportunity for improvement to better serve consumers                               213
Company believes com

## 4. Response Outcome and Timeliness Analysis

Investigate the relationship between the company's response
outcome and whether the response was submitted on time.

In [7]:
pd.crosstab(
    df['Company response to consumer'],
    df['Timely response?'],
    margins=True
)

Timely response?,No,Yes,All
Company response to consumer,,,
Closed with explanation,1026,150921,151947
Closed with monetary relief,181,23834,24015
Closed with non-monetary relief,49,10761,10810
Untimely response,117,0,117
All,1373,185516,186889


## 6. Duplicate detection
There are two different types of duplicates we need to investigate.

In [8]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481


In [22]:
def duplicate():
    print('-'*40)
    print(df.duplicated().sum())
    print(df['Complaint ID'].nunique())
    print('-'*40)


duplicate()


----------------------------------------
0
186889
----------------------------------------


## 7. Date quality investigation

In [39]:
columns = [
    'Date received',
    'Date sent to company'
]

for col in columns:
    temp_col = pd.to_datetime(df[col], errors='coerce', utc=True)

    print('-'*40)
    print(col)
    print(f'earliest complaint: {temp_col.min()}')
    print(f'latest complaint: {temp_col.max()}')
    print(temp_col.isna().sum())

    if temp_col.isna().sum() == 0:
        print('There are no invalid and unparseable dates','\n')
    else:
        print('Dates need to be checked','\n')
    print('-'*40)



----------------------------------------
Date received
earliest complaint: 2023-01-01 01:01:41+00:00
latest complaint: 2025-12-31 23:58:31+00:00
0
There are no invalid and unparseable dates 

----------------------------------------
----------------------------------------
Date sent to company
earliest complaint: 2023-01-01 01:12:26+00:00
latest complaint: 2026-08-18 10:45:46+00:00
0
There are no invalid and unparseable dates 

----------------------------------------


In [65]:
date_received = pd.to_datetime(df['Date received'], errors='coerce', utc=True)
date_sent = pd.to_datetime(df['Date sent to company'], errors='coerce', utc=True)

df['processing_delay_days'] = (date_sent - date_received).dt.days


def date():
    print('-'*40)
    print(df['processing_delay_days'].describe(),'\n')

    print(f'Negative dates: {(df['processing_delay_days'] < 0).sum()}')
    print(f'Maximum delay: {df['processing_delay_days'].max()}')
    print(f'Averaeg delay: {df['processing_delay_days'].mean()}')
    print(f'Number of records took more than 7 days: {(df['processing_delay_days'] > 7).sum()} ')
    print(f'Number of records took more than 30 days: {(df['processing_delay_days'] > 30).sum()}')
    print('-'*40)


date()

----------------------------------------
count    186889.000000
mean          0.971218
std           5.738789
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         269.000000
Name: processing_delay_days, dtype: float64 

Negative dates: 0
Maximum delay: 269
Averaeg delay: 0.971218209739471
Number of records took more than 7 days: 5830 
Number of records took more than 30 days: 2067
----------------------------------------
